In [20]:
# Import the libraries

%pip install yfinance 
%pip install plotly
%pip install scikit-learn

import yfinance as yf

import pandas as pd 
import numpy as np 

import plotly.graph_objects as go 
import plotly.express as px 
#import plotly.io as pio



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [32]:
# Download the stock data from yfinance
import yfinance as yf
# Define the stock ticker and the date range
tickers = ['AAPL']
#tickers.append('GC=F')  # Gold futures
#tickers.append('BTC-USD')  # Bitcoin in USD
start_date = '2020-01-01'
end_date = '2026-01-01'

# Download the stock data
stocks = yf.download(tickers, start=start_date, end=end_date, auto_adjust=True)
gold = yf.download('GC=F', start=start_date, end=end_date, auto_adjust=True)
BTC = yf.download('BTC-USD', start=start_date, end=end_date, auto_adjust=True)

# Arrange the data so that 'Close' price can directly be used for plotting and calculations
stocks.columns = stocks.columns.droplevel(1)  # Drop the ticker level from columns
gold.columns = gold.columns.droplevel(1)
BTC.columns = BTC.columns.droplevel(1)

# Merge the normalized closing price of gold and bitcoin with the stock data
#stocks['Gold'] = (gold['Close'] - np.mean(gold['Close'])) / np.std(gold['Close'])
#stocks['Bitcoin'] = (BTC['Close'] - np.mean(BTC['Close'])) / np.std(BTC['Close'])

# reset the index to make 'Date' a column
#stocks.reset_index(inplace=True)
# Drop the Open, High, Low columns
stocks.drop(columns=['Open', 'High', 'Low'], inplace=True)

stocks.head()

[*********************100%***********************]  1 of 1 completed


[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed


Price,Close,Volume
Date,,
2020-01-02,72.400520,135480400
2020-01-03,71.696632,146322800
2020-01-06,72.267921,118387200
2020-01-07,71.928055,108872000
2020-01-08,73.085091,132079200


In [33]:
# Normalize the gold and bitcoin prices to the first value to compare relative changes
def normalize_to_first(df):
    return df / df.iloc[0]

stocks['Gold'] = normalize_to_first(gold['Close'])
stocks['Bitcoin'] = normalize_to_first(BTC['Close'])
#stocks[['Gold', 'Bitcoin']] = stocks[['Gold', 'Bitcoin']].apply(normalize_to_first)

stocks.head(20)

Price,Close,Volume,Gold,Bitcoin
Date,,,,
2020-01-02,72.400520,135480400,1.000000,0.970181
2020-01-03,71.696632,146322800,1.016202,1.020098
2020-01-06,72.267921,118387200,1.027353,1.079032
2020-01-07,71.928055,108872000,1.031027,1.133819
2020-01-08,73.085091,132079200,1.021581,1.122176
2020-01-09,74.637497,170108400,1.017842,1.094289
2020-01-10,74.806229,140644800,1.021646,1.134216
2020-01-13,76.404427,121532000,1.015677,1.131111
2020-01-14,75.372711,161954400,1.011742,1.226049


In [34]:
### Add Bollinger Bands to the stock data
# Calculate the 20-day moving average and standard deviation


def Bollinger_Bands(df, window=20, num_std_dev=2):
    df['MA20'] = df['Close'].rolling(window=window).mean()
    df['STD20'] = df['Close'].rolling(window=window).std()
    df['UpperBand'] = df['MA20'] + (df['STD20'] * num_std_dev)
    df['LowerBand'] = df['MA20'] - (df['STD20'] * num_std_dev)
    return df
stocks = Bollinger_Bands(stocks)  
#stocks.dropna(inplace=True)
#.reset_index(inplace=True)  # Drop rows with NaN values resulting from rolling calculations

stocks.head()

# Plot the stock price and Bollinger Bands
#pio.renderers.default = "browser"
def plot_bollinger_bands(stocks):
    fig = go.Figure().update_layout(width=1000, height=600, title='AAPL Stock Price with Bollinger Bands', xaxis_title='Date', yaxis_title='Price (USD)')
    fig.add_trace(go.Scatter(x=stocks.index, y=stocks['Close'], mode='lines', name='Close Price'))

    fig.add_trace(go.Scatter(x=stocks.index, 
                            y=stocks['UpperBand'], 
                            mode = 'lines', 
                            name='Upper Bollinger Band', 
                            line=dict(color='red', dash='dot')))

    fig.add_trace(go.Scatter(x=stocks.index, 
                            y=stocks['LowerBand'], 
                            mode = 'lines', 
                            name='Lower Bollinger Band', 
                            line=dict(color='red', dash='dot')))
    fig.show()


In [39]:
# Get 20 day, 50 day, and 200 day moving averages
stocks['MA50'] = stocks['Close'].rolling(window=50).mean()
stocks['MA200'] = stocks['Close'].rolling(window=200).mean()

# Get the lag features for the closing price
stocks['Lag1'] = stocks['Close'].shift(1)
stocks['Lag2'] = stocks['Close'].shift(2)

# Capture volume spikes
stocks['VolumeSpike'] = np.where(stocks['Volume'] > 
                                 (stocks['Volume'].rolling(window=20).mean() + 
                                  2*stocks['Volume'].rolling(window=20).std()), 1, 0)

# 
def RSI(df, period=14):
    delta = df['Close'].diff()
    gains = delta.where(delta > 0, 0)
    losses = -delta.where(delta < 0, 0)
    avg_gains = gains.rolling(window=period).mean()
    avg_losses = losses.rolling(window=period).mean()
    rs = avg_gains/avg_losses
    #print(rs)
    df['RSI'] = 100 - (100/(1 + rs))
    return df

RSI(stocks, 14)

# MACD - Moving Average Convergence Divergence
def MACD(df, short_window=12, long_window=26, signal_window=9):
    df['EMA12'] = df['Close'].ewm(span=short_window).mean()
    df['EMA26'] = df['Close'].ewm(span=long_window).mean()
    df['MACD'] = df['EMA12'] - df['EMA26']
    df['SignalLine'] = df['MACD'].ewm(span=signal_window).mean()
    df['MACD_Histogram'] = df['MACD'] - df['SignalLine']
    return df

MACD(stocks)
###stocks.dropna(inplace=True)
stocks.head()


Price,Close,Volume,Gold,Bitcoin,MA20,STD20,UpperBand,LowerBand,MA50,MA200,Lag1,Lag2,VolumeSpike,RSI,EMA12,EMA26,MACD,SignalLine,MACD_Histogram
Date,,,,,,,,,,,,,,,,,,,
2020-01-02,72.400520,135480400,1.000000,0.970181,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,72.400520,72.400520,0.000000,0.000000,0.000000
2020-01-03,71.696632,146322800,1.016202,1.020098,NaN,NaN,NaN,NaN,NaN,NaN,72.400520,NaN,0,NaN,72.019248,72.035040,-0.015792,-0.008774,-0.007019
2020-01-06,72.267921,118387200,1.027353,1.079032,NaN,NaN,NaN,NaN,NaN,NaN,71.696632,72.400520,0,NaN,72.116305,72.118712,-0.002407,-0.006164,0.003757
2020-01-07,71.928055,108872000,1.031027,1.133819,NaN,NaN,NaN,NaN,NaN,NaN,72.267921,71.696632,0,NaN,72.056882,72.065413,-0.008531,-0.006966,-0.001565
2020-01-08,73.085091,132079200,1.021581,1.122176,NaN,NaN,NaN,NaN,NaN,NaN,71.928055,72.267921,0,NaN,72.336243,72.301880,0.034362,0.005328,0.029034


In [40]:
stocks['VolumeSpike'].value_counts()

VolumeSpike
0    1414
1      94
Name: count, dtype: int64

In [31]:
# Add the seasonality features
stocks['DayOfWeek'] = stocks.index.dayofweek
stocks['Month'] = stocks.index.month
stocks['Quarter'] = stocks.index.quarter

stocks.head()

Price,Close,High,Low,Open,Volume,Gold,Bitcoin,MA20,STD20,UpperBand,...,RSI,EMA12,EMA26,MACD,SignalLine,MACD_Histogram,DayOfWeek,Month,Quarter,InterestRate
Date,,,,,,,,,,,,,,,,,,,,,
2020-01-02,72.400520,72.460784,71.156682,71.409785,135480400,1.000000,0.970181,NaN,NaN,NaN,...,NaN,72.400520,72.400520,0.000000,0.000000,0.000000,3,1,1,NaN
2020-01-03,71.696632,72.455950,71.472454,71.629138,146322800,1.016202,1.020098,NaN,NaN,NaN,...,NaN,72.019248,72.035040,-0.015792,-0.008774,-0.007019,4,1,1,NaN
2020-01-06,72.267921,72.306491,70.568495,70.819193,118387200,1.027353,1.079032,NaN,NaN,NaN,...,NaN,72.116305,72.118712,-0.002407,-0.006164,0.003757,0,1,1,NaN
2020-01-07,71.928055,72.533095,71.708695,72.277578,108872000,1.031027,1.133819,NaN,NaN,NaN,...,NaN,72.056882,72.065413,-0.008531,-0.006966,-0.001565,1,1,1,NaN
2020-01-08,73.085091,73.386408,71.631537,71.631537,132079200,1.021581,1.122176,NaN,NaN,NaN,...,NaN,72.336243,72.301880,0.034362,0.005328,0.029034,2,1,1,NaN


In [27]:
def AddFEDFeatures(stocks, start_date, end_date):
    # Use the FED data to get the following features:
    # - Federal Funds Rate
    # - Inflation Rate
    # - Unemployment Rate
    %pip install fredapi
    from fredapi import Fred
    fred = Fred(api_key='c06a7676bffb4c8338823496ee6287bf')

    interest_rate = fred.get_series('FEDFUNDS', observation_start=start_date, observation_end=end_date)
    inflation_rate = fred.get_series('CPIAUCSL', observation_start=start_date, observation_end=end_date)
    unemployment_rate = fred.get_series('UNRATE', observation_start=start_date, observation_end=end_date)

    print(interest_rate.head())

    stocks['FederalFundsRate'] = interest_rate
    stocks['InflationRate'] = inflation_rate
    stocks['UnemploymentRate'] = unemployment_rate

